# Vedaz AI Astrologer - Fine-tuning Qwen2.5 with QLoRA

**Model:** Qwen/Qwen2.5-1.5B-Instruct  
**Method:** QLoRA (4-bit quantization + LoRA adapters)  
**GPU needed:** T4 15GB (free Google Colab)  
**Time:** ~20-40 minutes on T4

---

> **Before running:** Go to `Runtime -> Change runtime type -> T4 GPU`

## Step 0 - Check GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('GPU detected:', gpu)
    print('VRAM:', round(mem, 1), 'GB')
else:
    print('No GPU! Go to Runtime -> Change runtime type -> T4 GPU')


## Step 1 - Install Dependencies

Installs: transformers, trl (SFTTrainer), peft (LoRA), bitsandbytes (4-bit), datasets, huggingface_hub, accelerate

In [ ]:
!pip install -q transformers==4.44.0 trl==0.10.1 peft==0.12.0 bitsandbytes==0.43.3 datasets==2.21.0 huggingface_hub accelerate


## Step 2 - Upload Dataset

Upload `vedaz_finetune_ready.jsonl` (output of convert_to_jsonl.py)  
OR upload the original `Chat Data for assessment of applicants.json` directly.

In [ ]:
from google.colab import files
import json
print('Please upload your dataset file...')
uploaded = files.upload()
uploaded_filename = list(uploaded.keys())[0]
print('Uploaded:', uploaded_filename)


## Step 3 - Load and Prepare Dataset

- Handles both .json and .jsonl format automatically
- Parses mixed format (compact + pretty-printed)
- Validates each conversation structure
- Converts to HuggingFace Dataset

In [ ]:
from datasets import Dataset
import json

def load_mixed_json(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()
    objects, depth, start = [], 0, None
    for i, ch in enumerate(raw):
        if ch == '{':
            if depth == 0: start = i
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0 and start is not None:
                try: objects.append(json.loads(raw[start:i+1]))
                except json.JSONDecodeError: pass
                start = None
    return objects

def normalize(obj):
    messages = obj.get('messages', [])
    if not messages: return None
    clean = []
    for msg in messages:
        if msg.get('role') not in ('system','user','assistant'): return None
        if not str(msg.get('content','')).strip(): return None
        clean.append({'role': msg['role'], 'content': msg['content']})
    if clean[0]['role'] != 'system': return None
    if clean[-1]['role'] != 'assistant': return None
    return {'messages': clean}

print('Loading:', uploaded_filename)
if uploaded_filename.endswith('.jsonl'):
    raw_objects = []
    with open(uploaded_filename,'r',encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try: raw_objects.append(json.loads(line))
                except json.JSONDecodeError: pass
else:
    raw_objects = load_mixed_json(uploaded_filename)

print('Raw objects found:', len(raw_objects))
cleaned = [normalize(obj) for obj in raw_objects]
cleaned = [c for c in cleaned if c is not None]
print('Valid conversations:', len(cleaned))
print('Skipped (invalid):', len(raw_objects)-len(cleaned))
dataset = Dataset.from_list(cleaned)
print('Dataset ready:', len(dataset), 'examples')
print('\nSample (first 2 messages):')
for msg in dataset[0]['messages'][:2]:
    print(' ', msg['role'].upper(), ':', msg['content'][:120], '...')


## Step 4 - HuggingFace Login

Get your token at huggingface.co/settings/tokens -> New token -> Write access

In [ ]:
from huggingface_hub import login
login()  # Paste your HuggingFace Write token when prompted


## Step 5 - Load Qwen2.5-1.5B in 4-bit (QLoRA)

**What is QLoRA?**
- Full Qwen2.5-1.5B needs ~3GB VRAM
- 4-bit quantization compresses it to ~1.2GB
- We only train small LoRA adapter layers (not the whole model)
- Result: fine-tuning works on free T4 GPU


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print('Loading', MODEL_NAME, '...')
print('Downloading ~1.5GB, takes 2-3 minutes on first run...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

params_m = model.num_parameters() / 1e6
allocated_gb = torch.cuda.memory_allocated() / 1e9
print('Model loaded!', round(params_m), 'M parameters')
print('GPU memory used:', round(allocated_gb, 2), 'GB')


## Step 6 - Configure LoRA Adapters

**What is LoRA?**
- Instead of updating all 1.5B weights, insert tiny adapter layers
- Only ~1-2% of parameters get trained
- Adapter saved as ~50-100MB file after training
- Base model stays frozen


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = round(100 * trainable / total, 2)
print('LoRA configured!')
print('Trainable params:', round(trainable/1e6, 2), 'M /', round(total/1e6, 2), 'M total (', pct, '%)')


## Step 7 - Train!

Config:
- `num_train_epochs=3` - 3 passes over data
- `per_device_train_batch_size=2` with `gradient_accumulation_steps=4` = effective batch 8
- `max_seq_length=2048`
- `learning_rate=2e-4`

**Estimated time on T4: 25-40 minutes**


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = './vedaz-qwen25-astrologer'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    logging_steps=10,
    save_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    max_seq_length=2048,
    packing=False,
    dataset_text_field=None,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

print('Starting fine-tuning...')
print('Dataset:', len(dataset), 'conversations | Epochs: 3')
trainer.train()
print('Training complete!')


## Step 8 - Save LoRA Adapter

In [ ]:
import os
ADAPTER_DIR = './vedaz-qwen25-adapter'
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
total_size = sum(os.path.getsize(os.path.join(ADAPTER_DIR,f)) for f in os.listdir(ADAPTER_DIR))
print('Adapter saved to:', ADAPTER_DIR)
print('Total adapter size:', round(total_size/1e6, 1), 'MB')


## Step 9 - Merge and Save Full Model (for vLLM)

vLLM needs the merged model (base + adapter combined).


In [ ]:
from peft import AutoPeftModelForCausalLM
import torch

MERGED_DIR = './vedaz-qwen25-merged'
print('Merging LoRA adapter into base model... (takes 3-5 minutes)')

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print('Merged model saved to:', MERGED_DIR)
print('Ready for vLLM deployment!')


## Step 10 - Push to HuggingFace Hub

**Change YOUR_HF_USERNAME to your actual HuggingFace username before running.**


In [ ]:
# CHANGE THIS to your HuggingFace username
HF_USERNAME = 'subhi-tiwari77'
REPO_NAME = 'vedaz-astrologer-qwen25-1.5b'
FULL_REPO_ID = HF_USERNAME + '/' + REPO_NAME

print('Pushing to: https://huggingface.co/' + FULL_REPO_ID)

# Push adapter (lightweight)
trainer.model.push_to_hub(FULL_REPO_ID + '-adapter', private=True)
tokenizer.push_to_hub(FULL_REPO_ID + '-adapter', private=True)
print('Adapter pushed to huggingface.co/' + FULL_REPO_ID + '-adapter')

# Push merged model (for vLLM)
merged_model.push_to_hub(FULL_REPO_ID, private=True)
tokenizer.push_to_hub(FULL_REPO_ID, private=True)
print('Merged model pushed to huggingface.co/' + FULL_REPO_ID)
print('Share this link with the client: https://huggingface.co/' + FULL_REPO_ID)


## Step 11 - Test the Fine-tuned Model

In [ ]:
from transformers import pipeline

pipe = pipeline('text-generation', model=merged_model, tokenizer=tokenizer, device_map='auto')

tests = [
    [{'role':'system','content':'You are Vedaz AI Vedic astrologer.'},
     {'role':'user','content':'Meri sarkari naukri kab lagegi? DOB: 15 Aug 1998, 7AM, Patna'}],
    [{'role':'system','content':'You are Vedaz AI Vedic astrologer.'},
     {'role':'user','content':'Mujhe seene mein dard ho raha hai, kya ye kundli se related hai?'}],
]

for i, msgs in enumerate(tests, 1):
    print('--- Test', i, '---')
    print('User:', msgs[-1]['content'])
    out = pipe(msgs, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.9)
    resp = out[0]['generated_text'][-1]['content']
    print('AI:', resp[:500])
    print()


## Step 12 - Download Adapter Locally (Optional)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('vedaz_adapter', 'zip', ADAPTER_DIR)
print('Downloading vedaz_adapter.zip ...')
files.download('vedaz_adapter.zip')


---

## Summary

| Step | Result |
|------|--------|
| Dataset | Vedaz astrologer conversations loaded and validated |
| Base model | Qwen2.5-1.5B-Instruct loaded in 4-bit |
| Training | QLoRA fine-tuning, 3 epochs |
| Output | LoRA adapter + merged model |
| Deployed | HuggingFace Hub (private) |

**Model ready for vLLM deployment.** See vllm_hosting_guide.md
